# Evaluation: LLM Judge Scoring

Loads the raw inference results from Notebook 04 and scores every response with an LLM judge (Claude Haiku).
Output is `judge_scores.json` — the input for the analysis in Notebook 06.

**Pipeline position:** 01 Fine-Tuning → 02 Data Generation → 03 Student Distillation → 04 Inference → `[05 Judge Scoring]` → 06 Analysis<br>
**No GPU required.** Runs locally — reads from `results/` and calls the Anthropic API.

This notebook only scores. All aggregation, tables and statistics live in Notebook 06,
so that adding an analysis never requires paying for a new judge run.

## 1. Setup

Install dependencies and import standard libraries.

In [1]:
%%capture
!pip install -r ../requirements.txt

In [1]:
import json
import os
from datetime import date
import anthropic

RESULTS_DIR = "../results"

print(f"anthropic:   {anthropic.__version__}")
print(f"Results dir: {os.path.abspath(RESULTS_DIR)}")

anthropic:   0.86.0
Results dir: C:\Users\Battlestation\PycharmProjects\00_python-ki-advanced\distil-support-llm\results


## 2. Authentication

Loads `ANTHROPIC_API_KEY` from a local `.env` file.<br>
Create a `.env` file in the repo root with the following line if you haven't already:<br>
`ANTHROPIC_API_KEY=your_key_here`

In [2]:
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if ANTHROPIC_API_KEY:
    print(f"ANTHROPIC_API_KEY set. ({ANTHROPIC_API_KEY[:4]}...{ANTHROPIC_API_KEY[-4:]})")
else:
    print("WARNING: ANTHROPIC_API_KEY not set — add it to your .env file.")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("Anthropic client ready.")

ANTHROPIC_API_KEY set. (sk-a...GgAA)
Anthropic client ready.


## 3. Load Responses

Loads the three raw generation files into a single flat list.
Every entry carries its own `model_label`, so no per-model variables are needed
anywhere downstream.

`vram_measurements.json` is not loaded here — it holds no per-response data and
is only needed for the analysis in Notebook 06.

In [4]:
def load_json(filename):
    path = os.path.join(RESULTS_DIR, filename)
    with open(path, encoding="utf-8") as f:
        return json.load(f)


MODELS = ["base", "teacher", "student"]

entries = [
    {"model_label": label, **e}
    for label in MODELS
    for e in load_json(f"raw_generations_{label}.json")
]

n_queries = len({e["query_id"] for e in entries})

print(f"models:  {MODELS}")
print(f"queries: {n_queries}")
print(f"entries: {len(entries)}  ({n_queries} queries x {len(MODELS)} models)")

models:  ['base', 'teacher', 'student']
queries: 120
entries: 360  (120 queries x 3 models)


## 4. Set up judge

Blind scoring: the judge sees the query and the response, never the model label.
Four binary criteria, `temperature=0` for deterministic scoring, one retry on a JSON parse failure.

**`FRESH_RUN` matters.** The scoring sweep can resume an interrupted run by skipping
`(query_id, model_label)` pairs that are already scored. That is only safe as long as the
responses are unchanged. After re-running Notebook 04, the old scores belong to responses
that no longer exist — set `FRESH_RUN = True` once to discard them.

In [7]:
JUDGE_MODEL = "claude-haiku-4-5-20251001"

JUDGE_SYSTEM_PROMPT = """\
You are evaluating a customer support response written in German.
Score the response on 4 binary criteria. Each is 1 (met) or 0 (not met).

1. acknowledgement: Does the response open with empathy, recognition, or \
acknowledgement of the customer's situation (e.g., "Das tut mir leid", \
"Ich verstehe", "Vielen Dank für Ihre Anfrage")?
2. structured_steps: Does the response provide its main guidance as numbered \
steps, bullet points, or clearly enumerated items (NOT flowing prose)?
3. closing: Does the response end with an offer to help further, a follow-up \
invitation, or a clear professional closing?
4. tone: Is the overall tone professional, polite, and respectful (Sie-form, \
no rudeness, no overly casual language)?

Respond ONLY with valid JSON. No prose, no markdown:
{"acknowledgement": 0|1, "structured_steps": 0|1, "closing": 0|1, "tone": 0|1}"""

CRITERIA = ["acknowledgement", "structured_steps", "closing", "tone"]


def call_judge(query_text, response_text):
    user_prompt = (
        f"Customer query (for context):\n{query_text}\n\n"
        f"Response to evaluate:\n{response_text}"
    )
    for attempt in range(2):
        message = client.messages.create(
            model=JUDGE_MODEL,
            max_tokens=64,
            temperature=0,                  # deterministic scoring
            system=JUDGE_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_prompt}],
        )
        raw = message.content[0].text.strip()

        if "```" in raw:
            raw = raw.replace("```json", "").replace("```", "").strip()

        try:
            scores = json.loads(raw)
            scores["judge_total"] = sum(scores[k] for k in CRITERIA)
            return scores
        except json.JSONDecodeError:
            if attempt == 0:
                continue

    raise ValueError(f"Unparseable judge output after 2 attempts: {raw}")


print(f"Judge model: {JUDGE_MODEL}")
print("call_judge() ready.")

Judge model: claude-haiku-4-5-20251001
call_judge() ready.


## 5. Run LLM Judge

Scores all 180 responses across the three models using Claude Haiku as a blind judge.
Each call returns four binary scores (0 or 1): acknowledgement, structured steps, closing offer, and professional tone.

Run the sanity check cell first to spot any rubric issues before committing to 180 API calls.
Resumable: if the run is interrupted, re-running the sweep cell will skip already-scored entries.

In [8]:
import random

for entry in random.Random(42).sample(entries, 3):
    scores = call_judge(entry["query_text"], entry["response_text"])
    print(f"[{entry['model_label']}] {entry['query_id']}")
    print(f"Query:    {entry['query_text'][:150]}...")
    print(f"Response: {entry['response_text']}")
    print(f"Scores:   {scores}")
    print()

[student] q088
Query:    Ich moechte mein Konto endgueltig loeschen und sichergehen, dass wirklich alle meine Daten entfernt werden. Wie leite ich diese vollstaendige Loeschun...
Response: Ich verstehe Ihre Sorge — das ist natuerlich eine schwierige Entscheidung!

1. Bestaetigen Sie die Loesung der technischen Abwicklung
2. Entschuldigen Sie sich fuer die Unklarheiten — Vielen Dank fuer Ihre Anfrage
3. Informieren Sie sich über die Endgueltigkeit und die Sicherheit Ihrer Daten
4. Entschuldigen Sie sich fuer die Unklarheiten — Vielen Dank fuer Ihre Anfrage

Bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung.
Scores:   {'acknowledgement': 1, 'structured_steps': 1, 'closing': 1, 'tone': 0, 'judge_total': 3}

[base] q058
Query:    Der Rabattcode aus eurem Newsletter greift angeblich erst ab fuenfzig Euro Mindestbestellwert. Davon stand in der Mail kein Wort....
Response: Vielen Dank für Ihre Anfrage. Der Rabattcode in Ihrem Newsletter hat leider keinen Mindestbestellwert. Bitte konta

In [9]:
FRESH_RUN = False        # True discards existing scores — set once after re-running nb04

JUDGE_SCORES_PATH = os.path.join(RESULTS_DIR, "judge_scores.json")

if os.path.exists(JUDGE_SCORES_PATH) and not FRESH_RUN:
    with open(JUDGE_SCORES_PATH, encoding="utf-8") as f:
        judge_scores = json.load(f)
    scored = {(s["query_id"], s["model_label"]) for s in judge_scores}
    print(f"Resuming: {len(judge_scores)} existing scores, {len(entries) - len(scored)} remaining.")
else:
    judge_scores = []
    scored = set()
    print(f"Fresh run: scoring all {len(entries)} entries.")

for entry in entries:
    key = (entry["query_id"], entry["model_label"])
    if key in scored:
        continue

    scores = call_judge(entry["query_text"], entry["response_text"])

    judge_scores.append({
        "query_id":    entry["query_id"],
        "model_label": entry["model_label"],
        **{k: scores[k] for k in CRITERIA},
        "judge_total": scores["judge_total"],
    })
    scored.add(key)

    if len(judge_scores) % 20 == 0:
        print(f"  scored {len(judge_scores)}/{len(entries)}")

with open(JUDGE_SCORES_PATH, "w", encoding="utf-8") as f:
    json.dump(judge_scores, f, ensure_ascii=False, indent=2)

print(f"\nSaved: {JUDGE_SCORES_PATH}  ({len(judge_scores)} entries)")

Fresh run: scoring all 360 entries.
  scored 20/360
  scored 40/360
  scored 60/360
  scored 80/360
  scored 100/360
  scored 120/360
  scored 140/360
  scored 160/360
  scored 180/360
  scored 200/360
  scored 220/360
  scored 240/360
  scored 260/360
  scored 280/360
  scored 300/360
  scored 320/360
  scored 340/360
  scored 360/360

Saved: ../results\judge_scores.json  (360 entries)


## 6. Run Metadata

Records what produced these scores. `eval_summary.json` is written by Notebook 06,
not here — this notebook only produces judge output.

In [10]:
metadata = {
    "judge_model":       JUDGE_MODEL,
    "judge_temperature": 0,
    "evaluation_date":   date.today().isoformat(),
    "torch_seed_base":   42,
    "generation_params": {
        "temperature":    0.7,
        "top_p":          0.9,
        "max_new_tokens": 384,
        "do_sample":      True,
    },
    "n_queries":   n_queries,
    "n_responses": len(judge_scores),
}

with open(os.path.join(RESULTS_DIR, "eval_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved: eval_metadata.json")
print(json.dumps(metadata, indent=2))

Saved: eval_metadata.json
{
  "judge_model": "claude-haiku-4-5-20251001",
  "judge_temperature": 0,
  "evaluation_date": "2026-08-30",
  "torch_seed_base": 42,
  "generation_params": {
    "temperature": 0.7,
    "top_p": 0.9,
    "max_new_tokens": 384,
    "do_sample": true
  },
  "n_queries": 120,
  "n_responses": 360
}
